## 3. A grading assistant

Teachers in general have a lot of administrations to do and one of those things is grading. Can we create a simple grade assistant to assist a Swedish teacher in grading? This exercise focuses a lot in prompt engineering and afterwards to postprocess the output using Pydantic.



In [1]:
from google import genai
import pandas as pd
from pydantic import BaseModel,ValidationError

a) Go into this page with examples of students answers to a particular question. Copy some example texts and paste it into files with names like `student_text_1.txt`, `student_text_2.txt`.


b) Read these data into python and tell your LLM to grade them.


In [134]:
text_list = []
for i in range(1,3):
    with open (f"3_txt/student_{i}.txt", "r", encoding="utf-8") as file:
        text_list.append(file.read())
text_list

['Inför allas blickar\nEn av de större andledningarna att man är tal rädd är förmodligen att man är\nrädd för att folket som lyssnar på kommer göra narr av en. När man är ensam\npå en scen eller i klass rummet så är det väldigt jobbigt när det sitter ett flertal\nmänniskor som sitter och tittar på en. Jag känner mig inte så säker när jag står och\ngör en föreläsning eller en presentation. Känslan när man är framför människor\nsom man inte riktigt känner är ganska obehagligt, för man vill inte att dem ska\ntycka illa om en, eller tycka att man är konstig på något vis. Men jag tror att det\ngår att bli av med talrädslan, om man börjar i en mindre grupp och gör sig vann\nmed att prata med människor utan att bli rädd eller känna obehag.\nPeter Letmark skriver i Dagens Nyheter 2012-05-04 att det har att göra med\nevolutionära förklaringar, att man är rädd för ormar eller hundar kan ju vara\nen sak som sitter i ryggmärgen. Det kan bero på att man är rädd för att bli\nutstöt från gruppen. Let

In [135]:
from function import get_gemini
grades_simple = get_gemini(f"""Betygsätt dessa 2 texter.
        Texter:
        
        {text_list}""")
grades_simple

'Här är en bedömning av de två texterna:\n\n---\n\n### Övergripande Bedömning\n\nBåda texterna behandlar ämnet talrädsla utifrån egna resonemang och med stöd från Peter Letmarks artikel. Text 2 är generellt starkare vad gäller struktur, argumentation och språklig korrekthet, medan Text 1 har en mer personlig ton men lider av betydande språkliga brister.\n\n---\n\n### Text 1: "Inför allas blickar"\n\n**Styrkor:**\n*   **Personlig koppling:** Texten börjar med en tydlig personlig reflektion ("Jag känner mig inte så säker...") vilket gör ämnet relaterbart och engagerande.\n*   **Relevant källanvändning:** Författaren använder Peter Letmarks artikel för att bredda perspektivet med evolutionära och sociala förklaringar till talrädsla, vilket stärker argumentationen.\n*   **Egna insikter:** Författaren delar med sig av egna strategier för att övervinna rädslan (börja i mindre grupp, prata om något man brinner för, vikten av självförtroende).\n\n**Svagheter:**\n*   **Språkliga brister:** Dett


c) Prompt to get an output of fields proposed_grade, motivation and improvements.


In [138]:
prompt_grade_as_dict= f"""Betygsätt dessa 2 texter.
        proposed_grade ska enbart någon av dessa(A är högst, F är lägst samt icke-godkänt): A, B, C, D, E, F
        output ska vara enbart i detta json-format:
        {{
            "proposed_grade": "..",
            "motivation": "...",
            "improvements": "...",
            "proposed_grade": "..",
            osv...
        }}
        Texter:
        
        {text_list}
        
        """
grades_dict = get_gemini(prompt_grade_as_dict)
grades_dict

'```json\n[\n    {\n        "proposed_grade": "F",\n        "motivation": "Texten lider av omfattande språkfel (stavning, grammatik, ordföljd) som allvarligt försvårar läsningen och förståelsen. Till exempel \\"andledningarna\\" istället för \\"anledningarna\\", \\"tal rädd\\" istället för \\"talrädd\\", \\"gör sig vann\\" istället för \\"gör sig van\\", samt felaktig användning av pronomen och dubbla negationer. Den saknar också tydlig styckeindelning, vilket gör att argumentationen känns osammanhängande och svår att följa då alla tankar presenteras i ett enda långt stycke. Trots att en källa används, är integrationen av källan med de egna tankarna inte fullt utvecklad, och de sista meningarna känns något löst kopplade till resten av argumentationen. Det stora antalet fel i kombination med den bristande strukturen gör att texten inte når upp till en godkänd nivå.",\n        "improvements": "Fokusera på grundläggande stavning och grammatik; använd korrekturläsningsverktyg och be någon 

In [139]:
import json
grades_dict = grades_dict.strip("`json")
data = json.loads(grades_dict) 
data

[{'proposed_grade': 'F',
  'motivation': 'Texten lider av omfattande språkfel (stavning, grammatik, ordföljd) som allvarligt försvårar läsningen och förståelsen. Till exempel "andledningarna" istället för "anledningarna", "tal rädd" istället för "talrädd", "gör sig vann" istället för "gör sig van", samt felaktig användning av pronomen och dubbla negationer. Den saknar också tydlig styckeindelning, vilket gör att argumentationen känns osammanhängande och svår att följa då alla tankar presenteras i ett enda långt stycke. Trots att en källa används, är integrationen av källan med de egna tankarna inte fullt utvecklad, och de sista meningarna känns något löst kopplade till resten av argumentationen. Det stora antalet fel i kombination med den bristande strukturen gör att texten inte når upp till en godkänd nivå.',
  'improvements': 'Fokusera på grundläggande stavning och grammatik; använd korrekturläsningsverktyg och be någon annan läsa igenom texten. Styckeindela texten logiskt för att se


d) Now validate this with pydantic model


In [140]:
from typing import Literal
from pydantic import BaseModel, ValidationError
class Grades(BaseModel): 
    proposed_grade: Literal["A", "B", "C","D","E","F"]
    motivation: str
    improvements: str
    
validated_grades = []
try:
    for d in data: 
        validated_grades.append(Grades.model_validate(d))
except ValidationError as err:
    print(err) 
len(validated_grades), validated_grades 

(2,
 [Grades(proposed_grade='F', motivation='Texten lider av omfattande språkfel (stavning, grammatik, ordföljd) som allvarligt försvårar läsningen och förståelsen. Till exempel "andledningarna" istället för "anledningarna", "tal rädd" istället för "talrädd", "gör sig vann" istället för "gör sig van", samt felaktig användning av pronomen och dubbla negationer. Den saknar också tydlig styckeindelning, vilket gör att argumentationen känns osammanhängande och svår att följa då alla tankar presenteras i ett enda långt stycke. Trots att en källa används, är integrationen av källan med de egna tankarna inte fullt utvecklad, och de sista meningarna känns något löst kopplade till resten av argumentationen. Det stora antalet fel i kombination med den bristande strukturen gör att texten inte når upp till en godkänd nivå.', improvements='Fokusera på grundläggande stavning och grammatik; använd korrekturläsningsverktyg och be någon annan läsa igenom texten. Styckeindela texten logiskt för att sepa


e) Output a folder with the following txt files: proposed_grade.txt, motivation.txt and improvements.txt


In [143]:
import pandas as pd 
grade_dict = [g.model_dump() for g in validated_grades]

df_grades = pd.DataFrame(grade_dict)
df_grades

df_grades["proposed_grade"].to_csv("3_txt/proposed_grade.txt")
df_grades["motivation"].to_csv("3_txt/motivation.txt")
df_grades["improvements"].to_csv("3_txt/improvements.txt")


f) Go [into skolverket for Svenska 1](https://www.skolverket.se/undervisning/gymnasieskolan/program-och-amnen-i-gymnasieskolan/hitta-program-amnen-och-kurser-i-gymnasieskolan-gy11/amne?url=907561864%2Fsyllabuscw%2Fjsp%2Fsubject.htm%3FsubjectCode%3DSVE%26version%3D8%26tos%3Dgy&sv.url=12.5dfee44715d35a5cdfa92a3) and copy "Betygskriterier" for "Svenska 1". These are the criterias for the different grades. Paste this into a file called `criterias.txt`.


In [144]:
with open("3_txt/criterias.txt", "r", encoding="utf-8") as file: 
    criterias = file.read()
criterias[:200]

'Betygskriterier\nBetyget E\n\nEleven kan, i förberedda samtal och diskussioner, muntligt förmedla egna tankar och åsikter samt genomföra muntlig framställning inför en grupp. Detta gör eleven med viss sä'


g) Now repeat b)-e) but with the criterias in your prompt as well. Can you see any differences in the outputs?


- b) Read these data into python and tell your LLM to grade them.


- c) Prompt to get an output of fields proposed_grade, motivation and improvements.


In [145]:
from function import get_gemini
prompt_grade_as_dict= f"""Betygsätt dessa 2 texter.
        proposed_grade enbart någon av dessa(A är högst, F är inte godkänt): A, B, C, D, E, F och utgå utifrån dessa betygskriterier:
        {criterias}
        output ska vara enbart i detta json-format:
        {{
            "proposed_grade": "..",
            "motivation": "...",
            "improvements": "...",
            "proposed_grade": "..",
            osv...
        }}
        Texter:
        
        {text_list}
        
        """
output = get_gemini(prompt_grade_as_dict)
output

'```json\n[\n    {\n        "proposed_grade": "F",\n        "motivation": "Texten lider av frekventa och grundläggande brister i språkriktigheten, inklusive stavfel (t.ex. \\"andledningarna\\", \\"vann\\"), felaktiga sammansättningar (t.ex. \\"klass rummet\\", \\"själv förtroende\\", \\"jobb skäl\\"), grammatiska fel (t.ex. \\"utstöt från\\", dubbel negation) och bristande meningsbyggnad/interpunktion. Dessa brister försvårar läsbarheten och tydligheten avsevärt, vilket gör att texten inte \\"i huvudsak följer skriftspråkets normer för språkriktighet\\" som krävs för ett E. Dispositionen är inte tydligt urskiljbar, och elevens reflektion över källan är ytlig och begränsas till att enbart instämma.",\n        "improvements": "1. **Språkriktighet:** Fokusera på grundläggande stavning, korrekta sammansättningar och grammatiska regler. Läs igenom texten noga för att identifiera och korrigera fel. Använd verktyg för stavnings- och grammatikkontroll. 2. **Meningsbyggnad och interpunktion:** 

In [146]:
print(output)

```json
[
    {
        "proposed_grade": "F",
        "motivation": "Texten lider av frekventa och grundläggande brister i språkriktigheten, inklusive stavfel (t.ex. \"andledningarna\", \"vann\"), felaktiga sammansättningar (t.ex. \"klass rummet\", \"själv förtroende\", \"jobb skäl\"), grammatiska fel (t.ex. \"utstöt från\", dubbel negation) och bristande meningsbyggnad/interpunktion. Dessa brister försvårar läsbarheten och tydligheten avsevärt, vilket gör att texten inte \"i huvudsak följer skriftspråkets normer för språkriktighet\" som krävs för ett E. Dispositionen är inte tydligt urskiljbar, och elevens reflektion över källan är ytlig och begränsas till att enbart instämma.",
        "improvements": "1. **Språkriktighet:** Fokusera på grundläggande stavning, korrekta sammansättningar och grammatiska regler. Läs igenom texten noga för att identifiera och korrigera fel. Använd verktyg för stavnings- och grammatikkontroll. 2. **Meningsbyggnad och interpunktion:** Öva på att variera m

In [147]:
cleaned = output.strip("`json")
# output = output.strip("")
data = json.loads(cleaned)
print(data)
# print(output)

[{'proposed_grade': 'F', 'motivation': 'Texten lider av frekventa och grundläggande brister i språkriktigheten, inklusive stavfel (t.ex. "andledningarna", "vann"), felaktiga sammansättningar (t.ex. "klass rummet", "själv förtroende", "jobb skäl"), grammatiska fel (t.ex. "utstöt från", dubbel negation) och bristande meningsbyggnad/interpunktion. Dessa brister försvårar läsbarheten och tydligheten avsevärt, vilket gör att texten inte "i huvudsak följer skriftspråkets normer för språkriktighet" som krävs för ett E. Dispositionen är inte tydligt urskiljbar, och elevens reflektion över källan är ytlig och begränsas till att enbart instämma.', 'improvements': '1. **Språkriktighet:** Fokusera på grundläggande stavning, korrekta sammansättningar och grammatiska regler. Läs igenom texten noga för att identifiera och korrigera fel. Använd verktyg för stavnings- och grammatikkontroll. 2. **Meningsbyggnad och interpunktion:** Öva på att variera meningsbyggnaden och att använda kommatecken och punk

- d) Now validate this with pydantic model


In [149]:
from pydantic import BaseModel, ValidationError
from typing import Literal
class GradesCriterias(BaseModel): 
    proposed_grade: Literal["A", "B", "C", "D", "E", "F"]
    motivation: str 
    improvements: str

valid_grades_crits = []
for d in data: 
    try:
        valid_grades_crits.append(GradesCriterias.model_validate(d))
    except ValidationError as err: 
        print(err)
valid_grades_crits

[GradesCriterias(proposed_grade='F', motivation='Texten lider av frekventa och grundläggande brister i språkriktigheten, inklusive stavfel (t.ex. "andledningarna", "vann"), felaktiga sammansättningar (t.ex. "klass rummet", "själv förtroende", "jobb skäl"), grammatiska fel (t.ex. "utstöt från", dubbel negation) och bristande meningsbyggnad/interpunktion. Dessa brister försvårar läsbarheten och tydligheten avsevärt, vilket gör att texten inte "i huvudsak följer skriftspråkets normer för språkriktighet" som krävs för ett E. Dispositionen är inte tydligt urskiljbar, och elevens reflektion över källan är ytlig och begränsas till att enbart instämma.', improvements='1. **Språkriktighet:** Fokusera på grundläggande stavning, korrekta sammansättningar och grammatiska regler. Läs igenom texten noga för att identifiera och korrigera fel. Använd verktyg för stavnings- och grammatikkontroll. 2. **Meningsbyggnad och interpunktion:** Öva på att variera meningsbyggnaden och att använda kommatecken oc

In [150]:
grades_crit_dict = [grade.model_dump() for grade in valid_grades_crits]
df_grades_crit = pd.DataFrame(grades_crit_dict)
df_grades_crit

,proposed_grade,motivation,improvements
0,F,Texten lider av frekventa och grundläggande br...,1. **Språkriktighet:** Fokusera på grundläggan...
1,E,"Texten är sammanhängande och begriplig, och di...",1. **Språkriktighet:** En grundlig korrekturlä...


- e) Output a folder with the following txt files: proposed_grade.txt, motivation.txt and improvements.txt

In [151]:
df_grades_crit["improvements"].to_csv("3_txt/improvements_2.txt")
df_grades_crit["motivation"].to_csv("3_txt/motivation_2.txt")
df_grades_crit["proposed_grade"].to_csv("3_txt/proposed_grade_2.txt")

### Difference in grades from gemini, with criteras vs without criterias

In [153]:
# df_summary = 
# df_summary = df_grades["proposed_grade"] 
# df_summary = df_summary.rename("proposed_grade_without")
# # df_summary = df_grades_crit["proposed_grade"]
# pd.DataFrame(df_summary)
dict_summary = {
    "proposed_grade_without_criterias": df_grades["proposed_grade"],
    "proposed_grade_with_criterias": df_grades_crit["proposed_grade"],
    # "facit_from_chatgpt": ["E", "E","C","C","A"] 
    
}
df_summary = pd.DataFrame(dict_summary)
df_summary 

,proposed_grade_without_criterias,proposed_grade_with_criterias
0,F,F
1,B,E


h) Can you improve the output quality by providing few shot examples?
- done